In [1]:
import os
import json as _json
from pathlib import Path
from dataclasses import dataclass, field
from typing import cast
from typing import List, Optional, Dict, Any, Tuple, Union, Callable, Set
import pandas as pd
import polars as pl
import io
import numpy as np
from types import SimpleNamespace
from polars.testing import assert_frame_equal as pl_assert_frame_equal
print('pandas:', pd.__version__, ' polars:', pl.__version__)

/opt/anaconda3/envs/my_nlp_env/lib/python3.12/site-packages/pandas/core/computation/expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.7' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/opt/anaconda3/envs/my_nlp_env/lib/python3.12/site-packages/pandas/core/arrays/masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.7' currently installed).
  from pandas.core import (


pandas: 3.0.2  polars: 1.39.3


In [2]:
# ── Fixtures ────────────────────────────────────────────────────────────────

# --- collect_filter ---
sample = "sp1"

# --- collect_pivot ---
FIX_COLLECT_PIVOT_TRIMMED_MEAN = lambda x, p=0.1: float(np.mean(x))
FIX_COLLECT_PIVOT_APPRAISE_BINNED_BEFORE = pd.DataFrame({
    "gene": ["g1", "g1", "g2"],
    "sequence": ["seq1", "seq1", "seq2"],
    "sample": ["sp1", "sp1", "sp2"],
    "coverage": [1.0, 2.0, 0.0],
    "taxonomy": ["tax1", "tax1", "tax2"],
    "found_in": ["binA_protein,binB_protein", "binA_protein", "binC_protein"],
})
FIX_COLLECT_PIVOT_APPRAISE_BINNED_GEN = pl.from_pandas(FIX_COLLECT_PIVOT_APPRAISE_BINNED_BEFORE)

print("✅ Fixtures loaded")


✅ Fixtures loaded


In [3]:
# ── Before wrappers (verbatim pandas) ───────────────────────────────────────

def before_collect_filter(appraise_binned=None):
    if appraise_binned is None:
        appraise_binned = pd.DataFrame({"gene":["g1"],"sequence":["s1"],"sample":["sp1"],"coverage":["1.0"]})
    appraise_binned["sample"] = appraise_binned["sample"].str.replace("\.1$", "", regex=True)
    appraise_binned = appraise_binned[appraise_binned["sample"] == sample]
    return appraise_binned

def before_collect_pivot(trimmed_mean, appraise_binned=None):
    if appraise_binned is None:
        appraise_binned = pd.DataFrame({"gene":["g1"],"sequence":["s1"],"sample":["sp1"],"coverage":["1.0"]})
    appraise_binned["found_in"] = appraise_binned["found_in"].str.split(",")
    appraise_binned = appraise_binned.explode("found_in")
    appraise_binned["found_in"] = appraise_binned["found_in"].str.replace("_protein$", "", regex=True)

    trimmed_binned = (appraise_binned.groupby(["gene", "found_in"])["coverage"]
        .sum()
        .reset_index()
        .pivot(index="gene", columns="found_in", values="coverage")
        .reset_index()
        .melt(id_vars="gene")
        .fillna(0)
        .groupby("found_in")["value"]
        .apply(trimmed_mean)
        .reset_index()
        )
    reference_bins = set(trimmed_binned[trimmed_binned["value"] > 0]["found_in"].to_list())
    return reference_bins

<>:6: SyntaxWarning: invalid escape sequence '\.'
<>:6: SyntaxWarning: invalid escape sequence '\.'
/var/folders/bj/46g7cgnj5nj8tq01jz71cbm00000gn/T/ipykernel_30824/3009354154.py:6: SyntaxWarning: invalid escape sequence '\.'
  appraise_binned["sample"] = appraise_binned["sample"].str.replace("\.1$", "", regex=True)


In [4]:
# ── Generated wrappers (verbatim LLM-generated Polars) ──────────────────────

def gen_collect_filter(appraise_binned=None):
    if appraise_binned is None:
        appraise_binned = pl.DataFrame({"gene":["g1"],"sequence":["s1"],"sample":["sp1"],"coverage":["1.0"]})

    appraise_binned = appraise_binned.with_columns(
        pl.col("sample").str.replace(r"\.1$", "", literal=False).alias("sample")
    )
    appraise_binned = appraise_binned.filter(pl.col("sample") == sample)
    return appraise_binned

def gen_collect_pivot(trimmed_mean, appraise_binned=None):
    if appraise_binned is None:
        appraise_binned = pl.DataFrame({"gene":["g1"],"sequence":["s1"],"sample":["sp1"],"coverage":["1.0"]})

    appraise_binned = appraise_binned.with_columns(
        pl.col("found_in").str.split(",")
    ).explode("found_in").with_columns(
        pl.col("found_in").str.replace_all(r"_protein$", "")
    )

    trimmed_binned = (
        appraise_binned.group_by(["gene", "found_in"])
        .agg(pl.col("coverage").sum())
        .pivot(index="gene", on="found_in", values="coverage")
        .melt(id_vars="gene")
        .fill_null(0)
        .group_by("variable")
        .agg(pl.col("value").map_groups(lambda ss: trimmed_mean(ss[0]), return_dtype=pl.Float64).alias("value"))
        .rename({"variable": "found_in"})
    )

    reference_bins = set(
        trimmed_binned.filter(pl.col("value") > 0).get_column("found_in").to_list()
    )
    return reference_bins

In [5]:
# ── Comparison helper ───────────────────────────────────────────────────────
def _index_is_trivial(idx):
    # Unnamed + integer-valued covers both a fresh RangeIndex and the leftover
    # positional index after filtering/boolean-masking a RangeIndex-based frame
    # (pandas downgrades RangeIndex to a plain Int64Index on filter, but it's
    # still just leftover row positions, not real data). A set_index(...)
    # always carries the original column's name, so any genuinely meaningful
    # index is caught by the "name is not None" branch.
    return idx.name is None and pd.api.types.is_integer_dtype(idx.dtype)


def _to_pl(r):
    if isinstance(r, pl.DataFrame): return r
    if isinstance(r, pd.DataFrame): return pl.from_pandas(r.reset_index(drop=True) if _index_is_trivial(r.index) else r.reset_index())
    if isinstance(r, pd.Series): return pl.from_pandas(r.to_frame().reset_index(drop=True) if _index_is_trivial(r.index) else r.to_frame().reset_index())
    return None

def compare(before_result, gen_result, label, check_row_order=False):
    raw_label = str(label)
    label_parts = raw_label.strip().split()
    is_l3 = bool(label_parts and label_parts[0].upper() == "L3")
    layer = "L3" if is_l3 else "L2"
    kind = "edge" if is_l3 else "equivalence"
    if is_l3:
        label_parts = label_parts[1:]
        if label_parts and label_parts[0].lower() in ("edge", "branch"):
            label_parts = label_parts[1:]
        display_label = " ".join(label_parts)
    else:
        display_label = raw_label

    left  = _to_pl(before_result.collect() if isinstance(before_result, pl.LazyFrame) else before_result)
    right = _to_pl(gen_result.collect() if isinstance(gen_result, pl.LazyFrame) else gen_result)
    if left is None and right is None:
        print(f"⚠️  {layer} {kind} {display_label}: both sides non-DataFrame (no output to compare)")
        return
    if left is None or right is None:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — one side returned DataFrame, other did not")
        return
    left_cols, right_cols = set(left.columns), set(right.columns)
    if left_cols != right_cols:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — column sets differ (before-only={left_cols - right_cols}, gen-only={right_cols - left_cols})")
        return
    common = list(left.columns)
    try:
        pl_assert_frame_equal(left.select(common), right.select(common),
                              check_dtypes=False, check_row_order=check_row_order)
        print(f"✅ {layer} {kind} {display_label}: MATCH")
    except Exception as e:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — {e}")


In [6]:
# === Tests: collect_pivot ===

try:
    _r = gen_collect_pivot(FIX_COLLECT_PIVOT_TRIMMED_MEAN, FIX_COLLECT_PIVOT_APPRAISE_BINNED_GEN)
    print("✅ L1 smoke gen_collect_pivot: OK, type=", type(_r).__name__)
except Exception as _e:
    print(f"❌ L1 smoke gen_collect_pivot: {type(_e).__name__}: {_e}")

try:
    _rb = before_collect_pivot(FIX_COLLECT_PIVOT_TRIMMED_MEAN, FIX_COLLECT_PIVOT_APPRAISE_BINNED_BEFORE.copy())
    print("✅ L1 smoke before_collect_pivot: OK")
except Exception as _e:
    print(f"❌ L1 smoke before_collect_pivot: {type(_e).__name__}: {_e}")

try:
    _rb = before_collect_pivot(FIX_COLLECT_PIVOT_TRIMMED_MEAN, FIX_COLLECT_PIVOT_APPRAISE_BINNED_BEFORE.copy())
    _rg = gen_collect_pivot(FIX_COLLECT_PIVOT_TRIMMED_MEAN, FIX_COLLECT_PIVOT_APPRAISE_BINNED_GEN)
    if set(_rb) == set(_rg):
        print("✅ L2 equivalence collect_pivot set: MATCH")
    else:
        print(f"❌ L2 equivalence collect_pivot set: MISMATCH — before={_rb}, gen={_rg}")
except Exception as _e:
    print(f"❌ L2 equivalence collect_pivot: setup error — {type(_e).__name__}: {_e}")

try:
    _before_error = _gen_error = None
    _rb = _rg = None
    try:
        _rb = before_collect_pivot(FIX_COLLECT_PIVOT_TRIMMED_MEAN, FIX_COLLECT_PIVOT_APPRAISE_BINNED_BEFORE.head(0).copy())
    except Exception as _e:
        _before_error = _e
    try:
        _rg = gen_collect_pivot(FIX_COLLECT_PIVOT_TRIMMED_MEAN, FIX_COLLECT_PIVOT_APPRAISE_BINNED_GEN.head(0))
    except Exception as _e:
        _gen_error = _e
    if _before_error is not None and _gen_error is not None:
        print("✅ L3 edge collect_pivot empty: both sides reject empty input")
    elif _before_error is None and _gen_error is None and set(_rb) == set(_rg):
        print("✅ L3 edge collect_pivot empty: MATCH")
    else:
        print(f"❌ L3 edge collect_pivot empty: MISMATCH — before_error={_before_error}, gen_error={_gen_error}, before={_rb}, gen={_rg}")
except Exception as _e:
    print(f"❌ L3 edge collect_pivot empty: {type(_e).__name__}: {_e}")


❌ L1 smoke gen_collect_pivot: AttributeError: 'Expr' object has no attribute 'map_groups'
✅ L1 smoke before_collect_pivot: OK
❌ L2 equivalence collect_pivot: setup error — AttributeError: 'Expr' object has no attribute 'map_groups'
❌ L3 edge collect_pivot empty: MISMATCH — before_error=None, gen_error='Expr' object has no attribute 'map_groups', before=set(), gen=None


/var/folders/bj/46g7cgnj5nj8tq01jz71cbm00000gn/T/ipykernel_30824/2674195740.py:27: DeprecationWarning: `DataFrame.melt` is deprecated; use `DataFrame.unpivot` instead, with `index` instead of `id_vars` and `on` instead of `value_vars`
  .melt(id_vars="gene")
/var/folders/bj/46g7cgnj5nj8tq01jz71cbm00000gn/T/ipykernel_30824/2674195740.py:27: DeprecationWarning: `DataFrame.melt` is deprecated; use `DataFrame.unpivot` instead, with `index` instead of `id_vars` and `on` instead of `value_vars`
  .melt(id_vars="gene")
/var/folders/bj/46g7cgnj5nj8tq01jz71cbm00000gn/T/ipykernel_30824/2674195740.py:27: DeprecationWarning: `DataFrame.melt` is deprecated; use `DataFrame.unpivot` instead, with `index` instead of `id_vars` and `on` instead of `value_vars`
  .melt(id_vars="gene")
